# Lab 3: Heat conduction

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 2: Projectile motion](../02_projectile/Lab_2_Projectile_Motion.ipynb) · Next: [Lab 4: Navier–Stokes flow](../04_navier_stokes/Lab_4_Navier_Stokes.ipynb)

Find the temperature across a bar made from two materials. Use one network for each material and match temperature and heat flux where they meet. Then vary the first material's conductivity.

The input is position, with conductivity added in the parameterized run; the output is temperature. The ends are held at 0 and 100. With $D_1=10$ and $D_2=0.1$, the exact interface temperature is $100/101$, about 0.99, not 50.

Check both interface temperature and heat-flux errors. Why can the temperature slope jump while the heat flux stays continuous? Here $D$ denotes conductivity; this is a steady conduction problem, not transient diffusion with a specified heat capacity.

## Steady heat conduction in a composite bar

### Problem

Find the steady temperature in two joined materials with different thermal conductivities.

<center><img src="images/diffusion_bar_geometry.png" alt="Drawing" style="width: 600px;"/></center>

The bar extends from $x=0$ to $x=2$, with conductivity $D_1=10$ on the left half and $D_2=0.1$ on the right. The ends are held at $0$ and $100$, respectively. Two networks represent the temperatures $U_1$ and $U_2$ on either side of the interface at $x=1$.

With no internal heat source, each material satisfies:

$$
\begin{align}
\frac{d}{dx}\left( D_1\frac{dU_1}{dx} \right) = 0, && \text{when } 0<x<1 \\
\frac{d}{dx}\left( D_2\frac{dU_2}{dx} \right) = 0, && \text{when } 1<x<2 \\
\end{align}
$$

Heat flux and temperature are continuous at the interface $(x=1)$:
$$
\begin{align}
D_1\frac{dU_1}{dx} = D_2\frac{dU_2}{dx}, && \text{when } x=1 \\
U_1 = U_2, && \text{when } x=1 \\
\end{align}
$$

### Step 1: Geometry and coefficients

Sample the two intervals, $[0,1]$ and $[1,2]$, using `torch.rand`. For $D_1=10$, $D_2=0.1$, $T_a=0$, and $T_c=100$, the interface temperature is

$$T_b=\frac{T_c+(D_1/D_2)T_a}{1+D_1/D_2}.$$

The analytical solution is $T_1=xT_b+(1-x)T_a$ on the left and $T_2=(x-1)T_c+(2-x)T_b$ on the right.

### Step 2: PDEs, interface and two networks

```python
class Diffusion(PDE):
    def __init__(self, field="u", conductivity="D1"):
        self.dim = 1
        x = Symbol("x")
        u = Function(field)(x)
        d = Symbol(conductivity) if isinstance(conductivity, str) else conductivity
        self.equations = {"diffusion": -d * u.diff(x, 2)}


class DiffusionInterface(PDE):
    def __init__(self):
        self.dim = 1
        x, d1 = Symbol("x"), Symbol("D1")
        a, b = Function("u_1")(x), Function("u_2")(x)
        self.equations = {"temperature_jump": a - b,
                          "flux_jump": d1 * a.diff(x) - D2 * b.diff(x)}


class CompositeBar(torch.nn.Module):
    def __init__(self, cfg, parameterized=False):
        super().__init__()
        self.parameterized = parameterized
        self.left = mlp(2 if parameterized else 1, 1, cfg)
        self.right = mlp(2 if parameterized else 1, 1, cfg)

    def forward(self, x, d1):
        resistance = (D2 / d1 - .012) / .008
        inputs = torch.cat((x - 1, resistance), dim=1) if self.parameterized else x - 1
        return (TA + x * (100 * D2 / d1) * self.left(inputs),
                TC + (2 - x) * 100 * self.right(inputs))
```

The factors $x$ and $2-x$ impose only the prescribed endpoint temperatures. The networks still learn the fields and both interface conditions. The left temperature scale $100D_2/D_1$ balances the small temperature change in the highly conductive material; it is not the analytical interface temperature. The parameter feature is the normalized resistance ratio $D_2/D_1$.

### From heat conduction to a steady 1D problem

The heat equation is

$$\rho c_p T_t-\nabla\cdot(k\nabla T)-q=0,$$

where $\rho c_p$ is volumetric heat capacity, $k$ is thermal conductivity, and $q$ is a heat source per unit volume. This bar is at steady state ($T_t=0$), has no source ($q=0$), and varies only along $x$. In each material the equation reduces to $-D_i T_i^{\prime\prime}(x)=0$, an ordinary differential equation in space. The code uses `D1` and `D2` for conductivity; the networks have no time input.

For a transient problem with constant volumetric heat capacity, division by $\rho c_p$ gives $T_t-\nabla\cdot(D\nabla T)-Q=0$, with thermal diffusivity $D=k/(\rho c_p)$ and $Q=q/(\rho c_p)$. Conductivity and diffusivity are different quantities.

At $x=1$, temperature and physical heat flux $-D_i T_i^{\prime}$ are continuous. The slopes can differ because the conductivities differ. Check both `temperature_jumps` and `physical_flux_jumps`.

### Step 3: Boundary, interior and interface losses

```python
def loss_terms(model, physics, batch_size, device):
    xl = torch.rand(batch_size, 1, device=device, requires_grad=True)
    d1 = 5 + 20 * torch.rand_like(xl) if model.parameterized else torch.full_like(xl, 10.0)
    xr = (1 + xl.detach()).requires_grad_()
    ul, _ = model(xl, d1)
    _, ur = model(xr, d1)
    rl = physics[0].forward({"coordinates": xl, "u_1": ul, "D1": d1})["diffusion"]
    rr = physics[1].forward({"coordinates": xr, "u_2": ur})["diffusion"]
    xi = torch.ones_like(xl, requires_grad=True)
    ui, vi = model(xi, d1)
    interface = physics[2].forward({"coordinates": xi, "u_1": ui, "u_2": vi, "D1": d1})
    at_left, _ = model(torch.zeros_like(xl), d1)
    _, at_right = model(torch.full_like(xr, 2), d1)
    return {"physics": (rl / (100 * D2)).square().mean() + (rr / (100 * D2)).square().mean(),
            "boundary": ((at_left - TA) / (100 * D2 / d1)).square().mean() + ((at_right - TC) / 100).square().mean(),
            "interface_temperature": (interface["temperature_jump"] / 10).square().mean(),
            "interface_flux": (interface["flux_jump"] / (100 * D2)).square().mean()}
```

This excerpt shows the random Adam batch. The source also accepts fixed collocation points for L-BFGS. The boundary term verifies the exact endpoint transform. Both PDE residuals and the flux jump use the same physical flux scale $100D_2$; dividing the flux jump by $100D_1$ would hide large relative heat-flux errors.

### Step 4: Check the temperature and heat flux

`heldout_before/after` evaluates 201 points in each material, including the boundaries and both sides of the interface. `per_conductivity` reports each material's RMSE relative to its own reference temperature change, maximum temperature error, boundary error, and interface temperature/flux jumps. For the parameterized model it checks all 41 values $D_1=5,5.5,\ldots,25$, including endpoints and values between training parameters. `accuracy` requires every case to pass: left/right relative RMSE at most 5%/0.5%, maximum temperature error 0.5 K, boundary and interface temperature error 0.1 K, interface flux error 2% of the reference flux, and each material's PDE RMSE divided by $100D_2$ at most 0.01. The analytical solution is used only for evaluation.

### Step 5: Configuration

[Fixed-D1 configuration](source_code/conf/config.yaml) · [Parameterized configuration](source_code/conf/config_param.yaml). Both use 300 optimizer calls in FP32: 200 Adam calls followed by 100 L-BFGS calls. Each L-BFGS call permits up to 20 internal iterations with line search; `closure_evaluations` records the actual work. A smaller `--steps` value is an execution check and must still pass `accuracy` before its plots are treated as an accurate solution.

### Step 6: Train the models

Run [diffusion_bar.py](source_code/diffusion_bar.py). Adam warms up both networks with random collocation samples, then L-BFGS refines them on fixed points. The parameterized refinement uses a 17-by-17 grid in position and conductivity. Endpoint temperatures are imposed exactly; temperature and heat-flux continuity are learned through the interface losses. All model parameters and training tensors use FP32.

In [ ]:
import os
import sys
import subprocess
import uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Start_Here.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the bootcamp repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings, completed_output
LAB = ROOT / "01_labs/03_heat_conduction"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs"))).expanduser().resolve()
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "300"))  # Adam + L-BFGS optimizer calls
RUN_DIRS = {}
RUN_COMPLETED = {}
validate_settings(DEVICE, STEPS)
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
RUN_COMPLETED["diffusion_bar"] = False
OUTPUT = OUTPUT_BASE / ("diffusion_bar-" + uuid.uuid4().hex[:8])
RUN_DIRS["diffusion_bar"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/diffusion_bar.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["diffusion_bar"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

### Plot the temperature

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "diffusion_bar")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
for i, d1 in enumerate(data["D1"]):
    plt.plot(data["x"], data["reference"][i, :, 0], "--", label=f"analytical D1={d1:g}")
    plt.plot(data["x"], data["prediction"][i, :, 0], label=f"PINN D1={d1:g}")
plt.axvline(1, color="grey"); plt.xlabel("x"); plt.ylabel("temperature"); plt.legend(); plt.show()

## Parameterized heat conduction

Train one pair of networks for $D_1\in(5,25)$, rather than a separate pair for each conductivity. After training, changing the conductivity input gives a new temperature prediction without retraining.

Parameterized PINNs are discussed in [Sun et al.](https://arxiv.org/abs/1906.02382).

### Model inputs and sampling

Both networks take position and the normalized resistance ratio $D_2/D_1$ as input. Adam batches sample $D_1\sim U(5,25)$; L-BFGS uses a fixed grid including both endpoints. Since conductivity is constant within each material, the equation uses SymPy `Symbol("D1")` rather than a spatial function.

[diffusion_bar_parameterized.py](source_code/diffusion_bar_parameterized.py) trains this model. The plots show $D_1=5,10,25$; accuracy evaluation covers $D_1=5,5.5,\ldots,25$ and checks both materials separately.

The full-bar curves look similar because the right material has much lower conductivity ($D_2=0.1$). Most of the temperature rise occurs there. The exact interface temperature is $T_b=100D_2/(D_1+D_2)$: about 1.96, 0.99, and 0.40 for $D_1=5,10,25$. The second panel enlarges the left material using the same predictions, so these differences are visible. Matching PINN and analytical curves should overlap; different conductivity cases should separate in the enlarged view.

In [ ]:
RUN_COMPLETED["diffusion_bar_parameterized"] = False
OUTPUT = OUTPUT_BASE / ("diffusion_bar_parameterized-" + uuid.uuid4().hex[:8])
RUN_DIRS["diffusion_bar_parameterized"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/diffusion_bar_parameterized.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["diffusion_bar_parameterized"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "diffusion_bar_parameterized")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
x = data["x"].reshape(-1)
full_bar = np.ones(x.shape, dtype=bool)
left_material = (x >= 0) & (x <= 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for i, d1 in enumerate(data["D1"]):
    for ax, region in zip(axes, (full_bar, left_material)):
        ax.plot(x[region], data["reference"][i, region, 0], "--",
                color=f"C{i}", linewidth=2, label=f"Analytical D1={d1:g}")
        ax.plot(x[region], data["prediction"][i, region, 0],
                color=f"C{i}", linewidth=1, marker="o", markersize=3, markevery=20,
                label=f"PINN D1={d1:g}")
axes[0].set(title="Full bar", xlim=(0, 2))
axes[0].axvline(1, color="grey", linewidth=1, alpha=0.5)
axes[0].legend(fontsize=9)
axes[1].set(title="Left material (zoom)", xlim=(0, 1))
for ax in axes:
    ax.set(xlabel="x", ylabel="Temperature")
    ax.grid(alpha=0.2)
plt.show()

## Reload the saved model: change conductivity without training

Choose new values inside $5\le D_1\le25$. The next cell launches a fresh process, reloads the parameterized checkpoint, and evaluates temperature and heat flux. It does **not** call an optimizer. Change only `D1_VALUES` and rerun this cell to compare materials. The original checkpoint and training results stay unchanged.

The left-material zoom makes small temperature differences visible. Both sides of $x=1$ are evaluated independently: matching temperatures alone does not prove that heat flux $q=-D\,dT/dx$ is continuous.

In [ ]:
TRAINED_RUN = completed_output(RUN_DIRS, RUN_COMPLETED, "diffusion_bar_parameterized")
D1_VALUES = [7.5, 15.0, 22.5]  # Change these inputs; do not rerun training.
INFERENCE_RUN = OUTPUT_BASE / ("inference-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/diffusion_bar_parameterized.py"),
           "--mode", "eval", "--checkpoint", str(TRAINED_RUN / "model.pt"),
           "--device", DEVICE, "--output-dir", str(INFERENCE_RUN),
           "--d1", *[str(value) for value in D1_VALUES]]
subprocess.run(command, check=True, cwd=ROOT)
from ETC.runtime.lab_visualization import plot_bar_fields
plot_bar_fields(INFERENCE_RUN)
plt.show()

## Compare both materials in ParaView

Download and extract the ZIP. Open both `material_1.vtp` and `material_2.vtp` for one case, then click **Apply**. Color both by `temperature` with the **same fixed color range**, normally 0–100; `manifest.json` records the full range, including any overshoot. Apply **Plot Data** to each curve to compare `temperature` with `reference_temperature`, then `heat_flux` with `reference_heat_flux`.

Both files include $x=1$. Any mismatch there is a model error, not a gap that the exporter fills. The data comes from the saved model's inference run, not a replacement analytical solution.

In [ ]:
from ETC.runtime.lab_visualization import export_bar
from IPython.display import FileLink, display
archive = export_bar(INFERENCE_RUN)
display(FileLink(os.path.relpath(archive, Path.cwd()), result_html_prefix="Download ParaView ZIP: "))

## Compare training losses in TensorBoard

Each training run writes live events under `outputs/_tensorboard/`. Compare the fixed and parameterized runs using `training/physics`, `training/interface_temperature`, and `training/interface_flux`. The boundary loss is zero by construction because the model imposes the two endpoint temperatures exactly.

The horizontal axis counts optimizer calls; L-BFGS can evaluate its loss more than once in a call. `training/closure_evaluations` records that work. Curves show the first loss evaluation in each call; use the held-out metrics and flux plot to assess the final model.

Set `OPEN_TENSORBOARD=True` to view these curves through your authenticated Jupyter connection. If the proxy page returns 404, update the Launchable environment rather than exposing a public TensorBoard port.

In [ ]:
OPEN_TENSORBOARD = False
if OPEN_TENSORBOARD:
    from ETC.runtime.lab_visualization import tensorboard_link
    display(tensorboard_link(OUTPUT_BASE / "_tensorboard"))

### Next steps

How should the interface temperature $T_b$ change as $D_1$ increases? With the endpoint temperatures fixed, it should decrease. Compare the three curves, the held-out solution error, and both interface errors. Matching temperature alone does not ensure heat-flux continuity.

Why use two networks and an interface loss? Think about the slope change at $x=1$. Lab 4 moves from this steady problem to a time-dependent flow.

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 2: Projectile motion](../02_projectile/Lab_2_Projectile_Motion.ipynb) · Next: [Lab 4: Navier–Stokes flow](../04_navier_stokes/Lab_4_Navier_Stokes.ipynb)

--- 

Further reading: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.